# 02 · A scalar autodiff engine: the `Value` class

*Build Your Own Autograd: a Tiny Automatic Differentiation Engine*, notebook 2 of 4

In notebook 01 you wrote the graph down as a dictionary before computing anything. An autodiff engine records the graph **while your code runs**: every arithmetic operation returns a new object that remembers its inputs and knows how to send gradients back to them. In this notebook you build that object, a `Value` that wraps one float, and you write the backward rule of every operation it supports.

## What you will build

- `value_add` and `value_mul`: addition and multiplication that record the graph and attach a backward rule;
- `value_pow`, `value_exp`, `value_log` and `value_tanh`: the remaining operations of the engine, each with its own rule;
- a check of every rule against central differences, and one neuron back-propagated by hand, node by node.

**Learning objectives.** By the end you can:

1. explain what a `Value` stores (`data`, `grad`, `_children`, `_op`, `_backward` and `label`) and why;
2. use Python's operator methods (`__add__`, `__radd__`, …) so that ordinary expressions build a graph;
3. write a backward rule as a closure that adds *local derivative × `out.grad`* into each child's `grad`;
4. verify every rule numerically, and explain why the order of the `_backward` calls matters.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0:** the Python standard library only. The core cells use `math` and `random`, imported in the setup cell, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- There are no optional cells, nothing is downloaded, and nothing in this notebook is random.

## Setup

Run this cell first. It creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import math
import random

rng = random.Random(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    if isinstance(want, (int, float)) and isinstance(got, (int, float)):
        return math.isclose(got, want, rel_tol=1e-9, abs_tol=atol)
    return got == want


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-9):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

## 📦 Recap: what you built earlier

The hidden cell below re-defines, under their usual names, the two pieces from notebook 01 that this notebook uses: `central_difference(f, x, h)` and the checking helper `practice_with(probe, name, yours, reference, cases)`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the pieces from notebook 01 that this notebook uses, under their usual names.

def central_difference(f, x, h=1e-6):
    """Estimate f'(x) from two nearby values of f: (f(x + h) - f(x - h)) / (2h)."""
    return (f(x + h) - f(x - h)) / (2 * h)


def practice_with(probe, name, yours, reference, cases, atol=1e-9):
    """Run practice on probe(yours) and probe(reference); return the function to use from now on.

    A probe wraps a function so that it returns plain numbers, lists or dicts
    that practice can compare. You get back your own function once it passes,
    otherwise the reference, exactly as with practice.
    """
    your_probe, reference_probe = probe(yours), probe(reference)
    chosen = practice(name, your_probe, reference_probe, cases, atol=atol)
    return yours if chosen is your_probe else reference

## 1 · A number that remembers how it was made

A `Value` wraps one float together with five more pieces of information:

| attribute | meaning |
|---|---|
| `data` | the number itself, computed by the forward pass |
| `grad` | $\partial(\text{final output}) / \partial(\text{this value})$; it starts at `0.0` and the backward pass fills it in |
| `_children` | a tuple of the `Value`s this one was computed from: its arguments, empty for an input |
| `_op` | a short string naming the operation, such as `"+"`, used only for printing |
| `_backward` | a function with no arguments that sends this value's `grad` on to its children |
| `label` | an optional name, used only for printing |

In notebook 01 the dictionary *was* the graph. Here each `Value` points to its children, so the graph is the web of objects that the arithmetic creates, and following `_children` from the output reaches every input. (In this course a node's **children** are the values it was computed from; they sit below it in the drawings.)

**How the operators work.** Python turns `a + b` into `a.__add__(b)` and, when the left side is a plain number, `2 + a` into `a.__radd__(2)`. Each such method in the class below hands its work to a function such as `value_add(a, b)`, which **you** write in this notebook. Python looks that name up only when `a + b` actually runs, so as soon as a check cell rebinds `value_add` to your version, every `+` between `Value`s runs your code.

Negation, subtraction and division need no rules of their own: `-a` is `a * -1`, `a - b` is `a + (-b)` and `a / b` is `a * b**-1`, so they work as soon as addition, multiplication and powers do. `backward()` calls `backward_pass`, which you build in notebook 03; in this notebook you call each `_backward` yourself.

In [ ]:
class Value:
    """A number that remembers how it was computed, so that gradients can flow back through it."""

    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)              # the number itself
        self.grad = 0.0                      # d(final output) / d(this value), filled in by the backward pass
        self._children = tuple(children)     # the Values this one was computed from
        self._op = op                        # the operation that made it, for printing
        self._backward = _no_gradient_to_pass_on   # replaced by the operation that creates this Value
        self.label = label                   # an optional name, for printing

    def __repr__(self):
        name = f"{self.label} | " if self.label else ""
        return f"Value({name}data={self.data:.6g}, grad={self.grad:.6g})"

    # Each operator hands the work to a module-level function, looked up when it runs.
    def __add__(self, other):        # self + other
        return value_add(self, as_value(other))

    def __radd__(self, other):       # other + self, when other is a plain number
        return value_add(as_value(other), self)

    def __mul__(self, other):        # self * other
        return value_mul(self, as_value(other))

    def __rmul__(self, other):       # other * self
        return value_mul(as_value(other), self)

    def __pow__(self, exponent):     # self ** exponent, for a constant int or float exponent
        if isinstance(exponent, Value):
            raise TypeError("the exponent must be a plain int or float, not a Value")
        return value_pow(self, exponent)

    # These four are built from the three above, so they need no rules of their own.
    def __neg__(self):               # -self
        return self * -1.0

    def __sub__(self, other):        # self - other
        return self + (-as_value(other))

    def __rsub__(self, other):       # other - self
        return as_value(other) + (-self)

    def __truediv__(self, other):    # self / other
        return self * as_value(other) ** -1

    def __rtruediv__(self, other):   # other / self
        return as_value(other) * self ** -1

    def exp(self):
        return value_exp(self)

    def log(self):
        return value_log(self)

    def tanh(self):
        return value_tanh(self)

    def relu(self):
        return value_relu(self)

    def backward(self):
        """Fill in .grad for every Value this one depends on."""
        backward_pass(self)


def _no_gradient_to_pass_on():
    """The _backward of a Value that no operation produced (an input): there is nothing to pass on."""


def as_value(x):
    """Wrap a plain number in a Value; leave a Value unchanged."""
    return x if isinstance(x, Value) else Value(x)


def backward_pass(root):
    """Placeholder: you write the real backward pass in notebook 03."""
    raise NotImplementedError("backward() comes in notebook 03; here, call each node's _backward() yourself.")

In [ ]:
a = Value(2.0, label="a")
print(a)
print("children:", a._children, "  op:", repr(a._op))
a._backward()          # an input has nothing to pass on, so this does nothing
print(a)

## 2 · The pattern, worked for you: ReLU

Every operation follows the same four steps. Here they are for $\operatorname{relu}(a) = \max(0, a)$, whose local derivative is $1$ where $a > 0$ and $0$ elsewhere:

1. compute the forward value from the children's `data`;
2. create the output `Value`, recording its children and the operation;
3. define a function `_backward` that adds *local derivative × `out.grad`* into each child's `grad`;
4. attach it with `out._backward = _backward`, and return `out`.

The inner function is a **closure**: it is created inside `value_relu`, so it remembers `a` and `out` long after `value_relu` has returned. When the backward pass later calls `out._backward()`, `out.grad` holds the gradient that has arrived from above, and the closure passes it on. It multiplies by `out.grad` because of the chain rule, and it uses `+=` because a child may also receive gradient from other places; notebook 03 shows exactly what goes wrong with `=`.

In [ ]:
def value_relu(a):
    """relu(a) = max(0, a) as a new Value, with its backward rule attached."""
    out = Value(a.data if a.data > 0 else 0.0, (a,), "relu")      # steps 1 and 2

    def _backward():                                                # step 3
        slope = 1.0 if a.data > 0 else 0.0                          # d relu(a) / da
        a.grad += slope * out.grad

    out._backward = _backward                                       # step 4
    return out


a = Value(1.5, label="a")
r = a.relu()
r.grad = 2.0           # pretend the backward pass has delivered dL/dr = 2 to r
r._backward()          # r passes it on: dL/da = 1 × 2
print(r, a, sep="\n")

### How the checks work in this notebook

Your operations return `Value`s, which `practice` cannot compare directly. So each check runs your function through a **probe** that:

1. builds fresh input `Value`s from plain numbers and gives each a starting `grad` of **0.5**, as if another part of a larger graph had already sent gradient there;
2. calls your function, sets `out.grad = -2.0` (a gradient arriving from above) and calls `out._backward()` once;
3. reports `out.data`, the children by name, and every input's final `grad`.

So a correct rule ends with `a.grad = 0.5 + local derivative × (-2.0)`. A case written `(1.5, "same")` passes one node as both arguments, as in `a * a`. After each check, `compare_with_central_difference(op, *point)` prints the gradient your rule produces next to a central difference.

In [ ]:
HEAD_START = 0.5     # gradient that "another branch" has already left on each input
UPSTREAM = -2.0      # the gradient arriving at the output from above


def _children_names(out, named):
    """The children of out, by name and sorted, so that their order does not matter."""
    names = []
    for child in out._children:
        matches = [name for name, node in named.items() if node is child]
        names.append(matches[0] if matches else "?")
    return sorted(names)


def binary_probe(op):
    """Probe for op(a, b): numbers in, a dict of numbers out. b_data may be "same" for op(a, a)."""
    def run(a_data, b_data):
        a = Value(a_data)
        b = a if b_data == "same" else Value(b_data)
        a.grad = b.grad = HEAD_START
        out = op(a, b)
        out.grad = UPSTREAM
        out._backward()
        return {"out.data": out.data, "children": _children_names(out, {"a": a, "b": b}),
                "a.grad": a.grad, "b.grad": b.grad}
    return run


def unary_probe(op):
    """Probe for op(a), or op(a, constant) when a constant follows a_data in the case."""
    def run(a_data, *constants):
        a = Value(a_data)
        a.grad = HEAD_START
        out = op(a, *constants)
        out.grad = UPSTREAM
        out._backward()
        return {"out.data": out.data, "children": _children_names(out, {"a": a}), "a.grad": a.grad}
    return run


def compare_with_central_difference(op, *point, h=1e-6):
    """Print op's backward-rule gradient beside a central difference for each input; return the largest gap."""
    inputs = [Value(x) for x in point]
    out = op(*inputs)
    out.grad = 1.0
    out._backward()
    worst = 0.0
    for index, x in enumerate(point):
        def along(t):
            shifted = list(point)
            shifted[index] = t
            return op(*[Value(s) for s in shifted]).data
        numerical = central_difference(along, x, h)
        gap = abs(inputs[index].grad - numerical)
        worst = max(worst, gap)
        print(f"   input {index}: _backward gives {inputs[index].grad:+.8f}, "
              f"central difference {numerical:+.8f}, gap {gap:.1e}")
    return worst

## 3 · Addition

For $\text{out} = a + b$, both local derivatives are $1$: $\partial\,\text{out} / \partial a = \partial\,\text{out} / \partial b = 1$. So addition hands the gradient arriving from above to **both** children unchanged; it distributes gradient. In code, the closure adds `out.grad` into `a.grad` and into `b.grad`.

The rule only works because the closure remembers `a`, `b` and `out`. The demo shows that memory on its own: `make_counter` returns an inner function that still reaches the `Value` it was created with.

In [ ]:
def make_counter():
    count = Value(0.0, label="count")
    def bump():
        count.data += 1          # the inner function still sees count after make_counter has returned
        return count
    return bump


bump = make_counter()
bump()
print(bump())

### ✍️ Your turn: `value_add`

Write `value_add(a, b)`:

- **inputs:** two `Value`s, `a` and `b`;
- **output:** a new `Value` with `data = a.data + b.data`, children `(a, b)` and op `"+"`, whose `_backward` adds `out.grad` into `a.grad` and into `b.grad`.

Hints: copy the four steps of `value_relu`. The local derivative is 1 for both children, so the closure is `a.grad += out.grad` and `b.grad += out.grad`. Use `+=`, never `=`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_add_reference(a, b):
    out = Value(a.data + b.data, (a, b), "+")

    def _backward():
        a.grad += out.grad
        b.grad += out.grad

    out._backward = _backward
    return out

In [ ]:
def value_add(a, b):
    """a + b as a new Value, with a _backward that sends out.grad to a and to b."""
    # ✍️ Your code here (about six lines, like value_relu)
    raise NotImplementedError

In [ ]:
value_add = practice_with(binary_probe, "value_add", value_add, value_add_reference, cases=[
    (2.0, -3.0),
    (0.25, 4.0),
    (1.5, "same"),          # a + a: one node receives both contributions
])
gap = compare_with_central_difference(value_add, 2.0, -3.0)
print(Value(2.0) + Value(-3.0), 1 + Value(4.0), sep="\n")

**Expected behaviour.** ✅, then a gradient of exactly $1$ for both inputs, next to a central difference that agrees to about ten digits. In the probe's first case, `out.data` is $-1$ and both inputs end at $0.5 + 1 \times (-2) = -1.5$; for `a + a` the single input ends at $0.5 - 2 - 2 = -3.5$, because it receives both contributions. The last two lines show `+` working between two `Value`s and with a plain number on the left, through `__radd__`.

## 4 · Multiplication

For $\text{out} = a \cdot b$, $\partial\,\text{out} / \partial a = b$ and $\partial\,\text{out} / \partial b = a$: each child's local derivative is the **other** child's value, exactly as in notebook 01. So the closure adds `b.data * out.grad` into `a.grad` and `a.data * out.grad` into `b.grad`.

Now `a * a` really needs the `+=`: the one node receives $a \times \text{out.grad}$ twice, for a total of $2a \times \text{out.grad}$, the derivative of $a^2$.

In [ ]:
print("central difference of t * (-3) at t = 2:", central_difference(lambda t: t * -3.0, 2.0))
print("so d(a·b)/da is b = -3, the other factor")

### ✍️ Your turn: `value_mul`

Write `value_mul(a, b)`:

- **inputs:** two `Value`s, `a` and `b`;
- **output:** a new `Value` with `data = a.data * b.data`, children `(a, b)` and op `"*"`, whose `_backward` adds `b.data * out.grad` into `a.grad` and `a.data * out.grad` into `b.grad`.

Hints: the closure reads `a.data` and `b.data`, the values from the forward pass. Once `value_mul` works, `-a`, `a - b` and `3 * a` work too.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_mul_reference(a, b):
    out = Value(a.data * b.data, (a, b), "*")

    def _backward():
        a.grad += b.data * out.grad
        b.grad += a.data * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_mul(a, b):
    """a * b as a new Value, with a _backward that sends b·out.grad to a and a·out.grad to b."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
value_mul = practice_with(binary_probe, "value_mul", value_mul, value_mul_reference, cases=[
    (2.0, -3.0),
    (0.5, 4.0),
    (1.5, "same"),          # a * a: the derivative of a² is 2a
])
gap = compare_with_central_difference(value_mul, 2.0, -3.0)
x = Value(3.0)
print(x * 2, -x, x - 1, 10 - x, sep="\n")

**Expected behaviour.** ✅, then gradients of $-3$ and $2$ for $a = 2$, $b = -3$, each matching its central difference. In the probe, `a * a` with $a = 1.5$ ends at $0.5 + 2 \times 1.5 \times (-2) = -5.5$; with `=` instead of `+=` it would end at $-3$, and the check would show ❌. The last four lines are $6$, $-3$, $2$ and $7$: negation and subtraction already work, built from your two rules.

## 5 · Powers with a constant exponent

For $\text{out} = a^n$, where $n$ is a fixed int or float, $\partial\,\text{out} / \partial a = n\,a^{n-1}$. The exponent is a plain number, not a `Value`, so it gets no gradient and is not a child: the node has a single child. This one rule covers squares ($n = 2$), square roots ($n = 0.5$) and reciprocals ($n = -1$), and reciprocals are how the class divides: $a / b = a \cdot b^{-1}$.

Stay inside the domain: a negative base with a fractional exponent gives a *complex* number in Python, and $0$ to a negative power raises `ZeroDivisionError`.

In [ ]:
print("3 · 2² =", 3 * 2.0 ** 2, "  central difference of t**3 at t = 2:", central_difference(lambda t: t ** 3, 2.0))

### ✍️ Your turn: `value_pow`

Write `value_pow(a, exponent)`:

- **inputs:** a `Value` `a` and a plain int or float `exponent`;
- **output:** a new `Value` with `data = a.data ** exponent`, children `(a,)` and op `f"**{exponent}"`, whose `_backward` adds `exponent * a.data ** (exponent - 1) * out.grad` into `a.grad`.

Hints: mind the comma in `(a,)`: it makes a one-element tuple. `Value.__pow__` already rejects a `Value` exponent, so here `exponent` is always a number.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_pow_reference(a, exponent):
    out = Value(a.data ** exponent, (a,), f"**{exponent}")

    def _backward():
        a.grad += exponent * a.data ** (exponent - 1) * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_pow(a, exponent):
    """a ** exponent for a constant int or float exponent, with its backward rule."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
value_pow = practice_with(unary_probe, "value_pow", value_pow, value_pow_reference, cases=[
    (2.0, 3),
    (4.0, 0.5),            # a square root
    (-2.0, -1),            # the reciprocal of a negative number
])
gap = compare_with_central_difference(lambda a: value_pow(a, 3), 2.0)
print(Value(6.0) / Value(3.0), 1 / Value(4.0), Value(9.0) ** 0.5, sep="\n")

**Expected behaviour.** ✅, then $3 \cdot 2^2 = 12$ beside its central difference. In the probe, $2^3$ ends at $0.5 + 12 \times (-2) = -23.5$. The last three lines show division and a square root: $6 / 3 = 2$, $1 / 4 = 0.25$ and $\sqrt 9 = 3$. Division now works between `Value`s because it is multiplication by a power.

## 6 · The exponential

$\frac{d}{da} e^a = e^a$: the exponential is its own derivative, so the rule can reuse the forward result instead of computing $e^a$ again: the closure adds `out.data * out.grad` into `a.grad`. `math.exp` overflows above about $709$ and raises `OverflowError`.

In [ ]:
print("exp(0.7) =", math.exp(0.7), "  central difference of exp at 0.7:", central_difference(math.exp, 0.7))

### ✍️ Your turn: `value_exp`

Write `value_exp(a)`:

- **input:** a `Value` `a`;
- **output:** a new `Value` with `data = math.exp(a.data)`, children `(a,)` and op `"exp"`, whose `_backward` adds `out.data * out.grad` into `a.grad`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_exp_reference(a):
    out = Value(math.exp(a.data), (a,), "exp")

    def _backward():
        a.grad += out.data * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_exp(a):
    """exp(a) as a new Value; its derivative is its own output."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
value_exp = practice_with(unary_probe, "value_exp", value_exp, value_exp_reference, cases=[
    (0.7,),
    (0.0,),
    (-3.0,),
])
gap = compare_with_central_difference(value_exp, 0.7)
print(Value(1.0).exp())

**Expected behaviour.** ✅, then a gradient of $e^{0.7} \approx 2.01375$ beside the same central difference, and $e \approx 2.71828$ on the last line.

## 7 · The natural logarithm

$\frac{d}{da} \ln a = 1/a$, for $a > 0$. This rule needs the child's value rather than the output: the closure adds `out.grad / a.data` into `a.grad`. `math.log` raises `ValueError` for $a \le 0$. Logarithms appear in losses such as cross-entropy, where a predicted probability of exactly $0$ would break training.

In [ ]:
print("1/4 =", 1 / 4, "  central difference of log at 4:", central_difference(math.log, 4.0))

### ✍️ Your turn: `value_log`

Write `value_log(a)`:

- **input:** a `Value` `a` with `a.data > 0`;
- **output:** a new `Value` with `data = math.log(a.data)`, children `(a,)` and op `"log"`, whose `_backward` adds `out.grad / a.data` into `a.grad`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_log_reference(a):
    out = Value(math.log(a.data), (a,), "log")

    def _backward():
        a.grad += out.grad / a.data

    out._backward = _backward
    return out

In [ ]:
def value_log(a):
    """The natural logarithm of a (a.data > 0) as a new Value, with its backward rule."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
value_log = practice_with(unary_probe, "value_log", value_log, value_log_reference, cases=[
    (4.0,),
    (1.0,),
    (0.01,),               # near zero the slope is steep: 1/a = 100
])
gap = compare_with_central_difference(value_log, 4.0)
print(Value(math.e).log())

**Expected behaviour.** ✅, then a gradient of $1/4$ beside its central difference, and $\ln e = 1$ on the last line. In the probe, $a = 0.01$ ends at $0.5 + 100 \times (-2) = -199.5$: near zero the logarithm is very steep.

## 8 · tanh

$\frac{d}{da} \tanh a = 1 - \tanh^2 a = 1 - \text{out}^2$. As in notebook 01, the rule uses the node's **output**: the closure adds `(1 - out.data ** 2) * out.grad` into `a.grad`. Writing `1 - a.data ** 2` instead is a common bug, and the check catches it.

In [ ]:
print("1 - tanh(0.5)² =", 1 - math.tanh(0.5) ** 2, "  central difference:", central_difference(math.tanh, 0.5))

### ✍️ Your turn: `value_tanh`

Write `value_tanh(a)`:

- **input:** a `Value` `a`;
- **output:** a new `Value` with `data = math.tanh(a.data)`, children `(a,)` and op `"tanh"`, whose `_backward` adds `(1 - out.data ** 2) * out.grad` into `a.grad`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def value_tanh_reference(a):
    out = Value(math.tanh(a.data), (a,), "tanh")

    def _backward():
        a.grad += (1.0 - out.data ** 2) * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_tanh(a):
    """tanh(a) as a new Value; its derivative is 1 - out**2."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
value_tanh = practice_with(unary_probe, "value_tanh", value_tanh, value_tanh_reference, cases=[
    (0.5,),
    (-1.2,),
    (4.0,),                # saturated: almost no gradient gets through
])
gap = compare_with_central_difference(value_tanh, 0.5)

**Expected behaviour.** ✅, then a gradient of $1 - \tanh^2 0.5 \approx 0.786448$ beside its central difference. Every rule of the engine is now written and checked: addition, multiplication, powers, `exp`, `log`, `tanh` and the ReLU you were given.

## 🧪 Experiments

### 🧪 Try it: tanh saturation

In [ ]:
x0 = 0.5   # try 2.0, 5.0 and -10.0, then run this cell again
z = Value(x0, label="z")
n = z.tanh()
n.grad = 1.0
n._backward()
print(f"tanh({x0}) = {n.data:+.6f}; local gradient 1 - tanh² = {z.grad:.2e}")

**What to notice.** At $x = 2$ the local gradient is already down to about $0.07$, at $x = 5$ it is below $2 \times 10^{-4}$, and at $x = -10$ it is about $8 \times 10^{-9}$. A saturated tanh passes almost nothing back, however wrong its output is. Every gradient below it in the graph is multiplied by this small number, so a network whose tanh units start saturated learns very slowly.

### 🧪 Try it: leave the domain

In [ ]:
x0 = 4.0   # try 0.0 and -1.0, then run this cell again
try:
    print(Value(x0).log())
except ValueError as error:
    print(f"log({x0}) raised ValueError: {error}")

**What to notice.** For $x \le 0$ the forward pass fails before any `Value` is created, so the error points at the bad input rather than appearing later as a strange gradient. The wording of the message differs between Python versions. Real losses guard against this, for example by computing the logarithm of a probability from logits, so that the argument can never be exactly zero.

## 9 · Put it together: back-propagate a neuron by hand

Now combine the operations into one neuron with a squared error,

$$L = \big(\tanh(w_1 x_1 + w_2 x_2 + b) - 0.9\big)^2 ,$$

and back-propagate through it by calling each node's `_backward` yourself. The rule for the order is the one from notebook 01: a node may pass its gradient on only after it has received all of it, that is, after every node that uses it has run its `_backward`. Here that is simply the reverse of the order in which the nodes were created.

We write `n + (-0.9)` rather than `n - 0.9` so that every node has a name: `-` would add two unnamed helper nodes (a constant $-1$ and a multiplication), and we would have to remember to call their `_backward` too. Notebook 03 finds the order automatically, helper nodes included. First, a printing helper: `draw(root)` shows the graph below `root` as a tree, in the style of notebook 01's `draw_graph`.

In [ ]:
def draw(root):
    """Print the graph below root as a tree: label, operation, data and grad of every node.

    A node that has already been drawn appears again as one line marked "(again)".
    """
    drawn = set()

    def describe(node):
        name = node.label or "·"
        how = node._op or "input"
        return f"{name} [{how}]  data {node.data:+.4f}  grad {node.grad:+.4f}"

    def walk(node, prefix, last, top):
        branch = "" if top else ("└─ " if last else "├─ ")
        again = node in drawn and bool(node._children)
        print(prefix + branch + describe(node) + ("   (again)" if again else ""))
        if again:
            return
        drawn.add(node)
        inner = prefix if top else prefix + ("   " if last else "│  ")
        for index, child in enumerate(node._children):
            walk(child, inner, index == len(node._children) - 1, False)

    walk(root, "", True, True)

In [ ]:
x1, x2 = Value(2.0, label="x1"), Value(-1.0, label="x2")
w1, w2, b = Value(0.4, label="w1"), Value(-0.3, label="w2"), Value(-0.5, label="b")

p1 = w1 * x1
p2 = w2 * x2
s = p1 + p2
z = s + b
n = z.tanh()
d = n + (-0.9)          # not n - 0.9: see the note above
L = d ** 2
for node, name in [(p1, "p1"), (p2, "p2"), (s, "s"), (z, "z"), (n, "n"), (d, "d"), (L, "L")]:
    node.label = name

L.grad = 1.0                                   # dL/dL = 1 starts the backward pass
for node in [L, d, n, z, s, p2, p1]:           # the reverse of the order of creation
    node._backward()

draw(L)

In [ ]:
leaves = [x1, x2, w1, w2, b]
point = [leaf.data for leaf in leaves]


def loss_at(values):
    """The same L as a function of the five leaf values: a forward pass only."""
    vx1, vx2, vw1, vw2, vb = [Value(v) for v in values]
    return ((vw1 * vx1 + vw2 * vx2 + vb).tanh() + (-0.9)) ** 2


print(f"L = {L.data:.6f}")
worst = 0.0
for index, leaf in enumerate(leaves):
    def along(t):
        shifted = list(point)
        shifted[index] = t
        return loss_at(shifted).data
    numerical = central_difference(along, point[index])
    gap = abs(leaf.grad - numerical)
    worst = max(worst, gap)
    print(f"dL/d{leaf.label:<2}  by hand {leaf.grad:+.8f}   central difference {numerical:+.8f}   gap {gap:.1e}")

assert math.isclose(loss_at(point).data, L.data), "loss_at should rebuild the same L"
assert math.isclose(n.grad, 2 * d.data, rel_tol=1e-12), "dL/dn should be 2(n - 0.9)"
assert worst < 1e-6, "a backward rule disagrees with central differences"

**What to notice.** The neuron outputs $n \approx 0.537$, below the target $0.9$, so $L \approx 0.1317$, and every gradient agrees with its central difference to about ten digits. The signs say what training would do: $\partial L / \partial w_1 \approx -1.033$ is negative, so increasing $w_1$ (whose input $x_1 = 2$ is positive) raises $n$ towards the target. Seven `_backward` calls in the right order gave all five gradients at once; working out that order by hand, helper nodes included, is exactly the chore that notebook 03 automates.

## Exercises

### Exercise 1 · The logistic sigmoid

$\sigma(a) = 1 / (1 + e^{-a})$ has the derivative $\sigma'(a) = \sigma(a)\,(1 - \sigma(a))$, which, like tanh's, can be written using the output alone. Write `value_sigmoid(a)` as a **single** node with op `"sigmoid"` (not as a combination of `exp`, `+` and `/`).

In [ ]:
def value_sigmoid_reference(a):
    out = Value(1.0 / (1.0 + math.exp(-a.data)), (a,), "sigmoid")

    def _backward():
        a.grad += out.data * (1.0 - out.data) * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_sigmoid(a):
    """sigmoid(a) = 1 / (1 + exp(-a)) as one node; its derivative is out·(1 - out)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
value_sigmoid = practice_with(unary_probe, "value_sigmoid", value_sigmoid, value_sigmoid_reference, cases=[
    (0.0,),
    (2.0,),
    (-3.5,),
])
gap = compare_with_central_difference(value_sigmoid, 0.8)

### Exercise 2 · A power with a `Value` exponent

Our `__pow__` accepts only a constant exponent. Write `value_pow_value(a, b)` for $a^b$ where **both** are `Value`s and $a > 0$: $\partial(a^b)/\partial a = b\,a^{b-1}$ and $\partial(a^b)/\partial b = a^b \ln a$. Use op `"pow"` and children `(a, b)`. The last check case, $x^x$, needs both contributions added into one node.

In [ ]:
def value_pow_value_reference(a, b):
    out = Value(a.data ** b.data, (a, b), "pow")

    def _backward():
        a.grad += b.data * a.data ** (b.data - 1) * out.grad
        b.grad += out.data * math.log(a.data) * out.grad

    out._backward = _backward
    return out

In [ ]:
def value_pow_value(a, b):
    """a ** b for two Values, with a.data > 0, and the backward rule for both."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
value_pow_value = practice_with(binary_probe, "value_pow_value", value_pow_value, value_pow_value_reference, cases=[
    (2.0, 3.0),
    (0.5, -1.5),
    (1.5, "same"),          # x ** x: d/dx = x**x · (ln x + 1)
])
gap = compare_with_central_difference(value_pow_value, 2.0, 3.0)

### Exercise 3 · Open questions

1. In "Put it together", swap `z` and `n` in the backward loop, so that it reads `[L, d, z, n, s, p2, p1]`, and run the two cells again (the first one rebuilds the graph). Which gradients change, and why are they wrong?
2. `value_relu` uses a slope of 0 at exactly $a = 0$. What does a central difference give there, and why does the choice almost never matter in training?
3. Why does `value_log` read `a.data` in its closure, while `value_exp` and `value_tanh` read `out.data`? Could `value_log` be written with `out.data` instead?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| ❌ where `a.grad` is off by exactly the head start, 0.5 | `=` replaced the gradient already there | use `a.grad += …` |
| right only when `out.grad` is 1 | the closure does not multiply by `out.grad` | `a.grad += local * out.grad` |
| only the tanh check fails | the rule used the input: `1 - a.data ** 2` | use the output: `1 - out.data ** 2` |
| every gradient stays at 0 | the closure was never attached, or `out` was not returned | end with `out._backward = _backward` and `return out` |
| `TypeError: the exponent must be a plain int or float` | `x ** y` with a `Value` exponent | use a number, or `value_pow_value` from Exercise 2 |
| `TypeError: unsupported operand type(s) for ** or pow()` | `2 ** x`: the class has no `__rpow__` | write $2^x$ as `(x * math.log(2)).exp()` |
| `ValueError` from `log` | the logarithm of zero or a negative number | keep the input positive |
| a `TypeError` that mentions `complex` | a negative base with a fractional exponent | keep the base positive |
| `ZeroDivisionError` | zero to a negative power, for example dividing by a `Value` equal to 0 | avoid zero denominators |
| `OverflowError: math range error` | `exp` of a number above about 709 | keep inputs moderate |

## Summary

- A `Value` stores its number (`data`), its gradient (`grad`), the values it came from (`_children`), the operation (`_op`), a backward rule (`_backward`) and a `label`.
- Operator methods (`__add__`, `__radd__`, `__mul__`, `__rmul__`, `__pow__`) make ordinary expressions build the graph; negation, subtraction and division are built from them.
- Every operation follows one pattern: compute `data`, create `out` with its children, define a closure that adds *local derivative × `out.grad`* into each child's `grad`, and attach it.
- The rules: add $1$ and $1$; multiply $b$ and $a$; power $n a^{n-1}$; exp the output; log $1/a$; tanh $1 - \text{out}^2$.
- Calling the `_backward`s by hand works, but only in the right order: every node after all the nodes that use it.

You built `value_add`, `value_mul`, `value_pow`, `value_exp`, `value_log` and `value_tanh`.

## Next

In **03 · Reverse-mode autodiff** (`03_reverse_mode_autodiff.ipynb`) you automate the order with a topological sort and write `backward()`, so that one call fills in every gradient. You will also see exactly what breaks when a gradient is assigned instead of accumulated, and why gradients must be reset between training steps.